In [1]:
import pandas as pd
import json
import random
import shutil
from pathlib import Path
from typing import Dict, Any, Union, List, Optional, Set
from __future__ import annotations
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from collections import Counter

In [8]:
def sample_timing_images_stratified(
    dataset_folder,
    n_images: int = 100,
    output_root: Union[Path, str] = "./Timing Experiments Data",
    random_seed: int = 42,
    write_subset_coco: bool = True,
) -> Dict[str, Any]:

    dataset_folder = Path(dataset_folder)
    ann_dir = dataset_folder / "annotations"

    out_root = Path(output_root) / dataset_folder.name
    out_img_dir = out_root / "images"
    out_ann_dir = out_root / "annotations"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_ann_dir.mkdir(parents=True, exist_ok=True)

    # 1) Locate the COCO annotation file
    ann_candidates = list(ann_dir.glob("*.json"))
    if not ann_candidates:
        raise FileNotFoundError(f"No JSON annotations found in {ann_dir}")

    coco_path, coco = None, None
    for p in ann_candidates:
        try:
            obj = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(obj, dict) and {"images", "annotations", "categories"} <= obj.keys():
                coco_path, coco = p, obj
                break
        except Exception:
            continue

    if coco is None:
        raise ValueError("No valid COCO annotations file found.")

    def resolve_src_path(file_name: str) -> Path:
        p = Path(file_name)
        return p if p.is_absolute() else (coco_path.parent / p).resolve()

    images_all = coco.get("images", [])
    anns = coco.get("annotations", [])
    cats = coco.get("categories", [])
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

    image_entries = [im for im in images_all if im.get("file_name") and 
                     resolve_src_path(im["file_name"]).suffix.lower() in exts and 
                     resolve_src_path(im["file_name"]).is_file()]
    
    if not image_entries:
        raise ValueError("No matching images found on disk.")

    num_images = len(image_entries)
    n_pick = min(int(n_images), num_images)
    imgid_to_idx = {im["id"]: i for i, im in enumerate(image_entries)}
    cat_ids = sorted({c["id"] for c in cats if "id" in c})
    num_cats = len(cat_ids)
    rng = random.Random(random_seed)

    # 3) Stratified Selection
    if num_cats <= 1 or MultilabelStratifiedShuffleSplit is None:
        indices = list(range(num_images))
        rng.shuffle(indices)
        chosen_idx = indices[:n_pick]
    else:
        catid_to_col = {cid: j for j, cid in enumerate(cat_ids)}
        y = np.zeros((num_images, num_cats), dtype=int)
        for ann in anns:
            i = imgid_to_idx.get(ann.get("image_id"))
            cid = ann.get("category_id")
            if i is not None and cid in catid_to_col:
                y[i, catid_to_col[cid]] = 1

        if int(y.sum()) == 0:
            indices = list(range(num_images))
            rng.shuffle(indices)
            chosen_idx = indices[:n_pick]
        else:
            X = np.arange(num_images).reshape(-1, 1)
            msss = MultilabelStratifiedShuffleSplit(test_size=n_pick, random_state=random_seed)
            _, timing_idx = next(msss.split(X, y))
            chosen_idx = list(timing_idx)

    # ---- Shuffling the selection ----
    # This determines the order in the folder and the JSON
    chosen_images = [image_entries[i] for i in chosen_idx]
    rng.shuffle(chosen_images) 

    # 4) Copy images with Shuffled Index Prefix
    # This forces the OS folder to show them in the shuffled order
    copied = skipped_existing = missing_sources = 0
    final_images_list = []
    chosen_image_ids = set()

    for idx, im in enumerate(chosen_images):
        src = resolve_src_path(im["file_name"])
        # New name looks like: 00001_original_name.jpg
        new_name = f"{idx:05d}_{src.name}"
        dst = out_img_dir / new_name
        
        if not src.is_file():
            missing_sources += 1
            continue
        
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1
        else:
            skipped_existing += 1
        
        # Prepare the new image entry for the JSON
        new_im = dict(im)
        new_im["file_name"] = f"../images/{new_name}"
        final_images_list.append(new_im)
        chosen_image_ids.add(im["id"])

    # 5) Write subset COCO
    subset_json_path = None
    subset_ann_count = 0
    if write_subset_coco:
        subset_anns = [a for a in anns if a.get("image_id") in chosen_image_ids]
        subset_ann_count = len(subset_anns)
        subset_coco = {
            "info": coco.get("info", {}),
            "licenses": coco.get("licenses", []),
            "categories": cats,
            "images": final_images_list, # Already in shuffled order
            "annotations": subset_anns,
        }
        subset_json_path = out_ann_dir / "instances_timing.json"
        with subset_json_path.open("w", encoding="utf-8") as f:
            json.dump(subset_coco, f, ensure_ascii=False, indent=2)

    return {"status": "Complete", "copied": copied, "output": str(out_root)}

In [7]:
# Make the split
summary = sample_timing_images_stratified("../Data/tomatoes", n_images=100, random_seed=42)
print(summary)
summary = sample_timing_images_stratified("../Data/apples", n_images=100, random_seed=42)
print(summary)

{'status': 'Complete', 'copied': 100, 'output': 'Timing Experiments/tomatoes'}
{'status': 'Complete', 'copied': 100, 'output': 'Timing Experiments/apples'}


#Evaluate Results

In [9]:
def avg_annotation_time_per_object_batches(
    coco_path: Path | str,
    annotation_batches: dict,
) -> pd.DataFrame:
    """
    Computes batch statistics and the running Relative Standard Error (RSE)
    of the average annotation time per object.
    """
    coco = json.loads(Path(coco_path).read_text(encoding="utf-8"))
    images_all = coco["images"]
    anns_all = coco["annotations"]

    rows = []
    all_sec_per_obj = []

    # Ensure batches are processed in order to calculate running RSE correctly
    sorted_keys = sorted(annotation_batches.keys())

    for batch_key in sorted_keys:
        b = annotation_batches[batch_key]
        start_idx, end_idx = b["image_index_range"]
        images = images_all[start_idx : end_idx + 1]
        image_ids = {im["id"] for im in images}

        anns = [a for a in anns_all if a["image_id"] in image_ids]
        per_image = Counter(a["image_id"] for a in anns)

        # parse 'MM:SS.ms' -> seconds
        m, rest = b["annotation_time"].split(":")
        s, ms = rest.split(".")
        total_time_sec = int(m) * 60 + int(s) + int(ms) / 100

        num_objects = len(anns)
        avg_obj_per_img = (num_objects / len(images)) if len(images) else 0.0
        avg_sec_per_obj = (total_time_sec / num_objects) if num_objects else 0.0
        
        # Store for RSE calculation
        all_sec_per_obj.append(avg_sec_per_obj)
        
        # Calculate Running RSE (%)
        # RSE = (Standard Error / Mean) * 100
        # Standard Error = StdDev / sqrt(n)
        current_n = len(all_sec_per_obj)
        if current_n > 1:
            current_mean = np.mean(all_sec_per_obj)
            current_std = np.std(all_sec_per_obj, ddof=1) # Sample standard deviation
            standard_error = current_std / np.sqrt(current_n)
            rse = (standard_error / current_mean) * 100
        else:
            rse = 0.0 # Cannot calculate variance with only one data point

        rows.append(
            {
                "batch_key": batch_key,
                "image_index_range": b["image_index_range"],
                "total_objects": num_objects,
                "avg_objects_per_image": round(avg_obj_per_img, 2),
                "min_objects_in_image": min(per_image.values(), default=0),
                "max_objects_in_image": max(per_image.values(), default=0),
                "total_annotation_time": b["annotation_time"],
                "avg_seconds_per_object": round(avg_sec_per_obj, 2),
                "running_rse_pct": round(rse, 2),
            }
        )

    return pd.DataFrame(rows)

In [10]:
def _parse_mmss_ms_to_seconds(t: str) -> float:
    """
    Parse 'MM:SS.ms' into seconds.
    ms is treated as hundredths.
    """
    m, rest = t.split(":")
    s, ms = rest.split(".")
    return int(m) * 60 + int(s) + int(ms) / 100

def avg_removal_time_per_bbox_batches(
    coco_path: Path | str,
    correction_batches: dict,
) -> pd.DataFrame:
    """
    Builds a DataFrame with per-batch stats.
    Only batches with > 0 removals contribute to the running RSE calculation.
    """
    coco = json.loads(Path(coco_path).read_text(encoding="utf-8"))
    images_all = coco["images"]
    anns_all = coco["annotations"]

    rows = []
    all_removal_times = [] # This will only store valid removal actions
    
    sorted_keys = sorted(correction_batches.keys())

    for batch_key in sorted_keys:
        b = correction_batches[batch_key]
        start_idx, end_idx = b["image_index_range"]
        images = images_all[start_idx : end_idx + 1]
        image_ids = {im["id"] for im in images}

        anns = [a for a in anns_all if a["image_id"] in image_ids]
        per_image = Counter(a["image_id"] for a in anns)

        images_analysed = len(images)
        total_objects = len(anns)
        avg_objects_per_image = (total_objects / images_analysed) if images_analysed else 0.0

        n_initial = int(b["nr_anns_pre_rem"])
        n_after = int(b["nr_anns_post_rem"])
        removed = n_initial - n_after

        batch_review_time = (1/3) * n_initial 
        rse = 0.0

        if removed > 0:
            total_sec = _parse_mmss_ms_to_seconds(b["removal_time"])
            avg_removal_time = (total_sec - batch_review_time) / removed
            
            # Update running stats ONLY for active removal batches
            all_removal_times.append(avg_removal_time)
            current_n = len(all_removal_times)
            
            if current_n > 1:
                current_mean = np.mean(all_removal_times)
                current_std = np.std(all_removal_times, ddof=1)
                se = current_std / np.sqrt(current_n)
                rse = (se / current_mean) * 100 if current_mean != 0 else 0.0
        else:
            avg_removal_time = 0.0
            # If no removals, the RSE remains what it was in the previous valid batch
            if all_removal_times:
                current_mean = np.mean(all_removal_times)
                current_std = np.std(all_removal_times, ddof=1) if len(all_removal_times) > 1 else 0
                se = current_std / np.sqrt(len(all_removal_times))
                rse = (se / current_mean) * 100 if current_mean != 0 else 0.0

        rows.append(
            {
                "batch_key": batch_key,
                "image_index_range": b["image_index_range"],
                "total_objects": total_objects,
                "avg_objects_per_image": round(avg_objects_per_image, 2),
                "nr_anns_pre_rem": n_initial,
                "nr_anns_post_rem": n_after,
                "review_time_overhead": round(batch_review_time, 2),
                "avg_removal_time": round(avg_removal_time, 2),
                "running_rse_pct": round(rse, 2),
            }
        )

    return pd.DataFrame(rows)

## Evaluate results tomatoes

In [11]:
ANNOTATION_BATCHES = {
    1: {"image_index_range": (0, 9), "annotation_time": "6:29.43"},
    2: {"image_index_range": (10, 19), "annotation_time": "6:12.28"},
    3: {"image_index_range": (20, 29), "annotation_time": "5:53.61"},
    4: {"image_index_range": (30, 39), "annotation_time": "6:00.68"},
    5: {"image_index_range": (40, 49), "annotation_time": "5:59.12"},
}
df = avg_annotation_time_per_object_batches(Path("Timing Experiments Data/tomatoes/annotations/instances_timing.json"), ANNOTATION_BATCHES)
df

,batch_key,image_index_range,total_objects,avg_objects_per_image,min_objects_in_image,max_objects_in_image,total_annotation_time,avg_seconds_per_object,running_rse_pct
0,1,"(0, 9)",74,7.4,3,9,6:29.43,5.26,0.00
1,2,"(10, 19)",70,7.0,3,9,6:12.28,5.32,0.53
2,3,"(20, 29)",80,8.0,6,10,5:53.61,4.42,5.81
3,4,"(30, 39)",81,8.1,7,10,6:00.68,4.45,5.08
4,5,"(40, 49)",76,7.6,6,11,5:59.12,4.73,4.00


In [ ]:
CORRECTION_BATCHES = {
    5: {"image_index_range": (40, 49), "nr_anns_pre_rem": 82, "nr_anns_post_rem": 57, "removal_time": "1:26.56"},
    6: {"image_index_range": (50, 59), "nr_anns_pre_rem": 86, "nr_anns_post_rem": 51, "removal_time": "1:48.73"},
    7: {"image_index_range": (60, 69), "nr_anns_pre_rem": 101, "nr_anns_post_rem": 44, "removal_time": "2:17.81"},
    8: {"image_index_range": (70, 79), "nr_anns_pre_rem": 90, "nr_anns_post_rem": 50, "removal_time": "1:41.44"},
    9: {"image_index_range": (80, 89), "nr_anns_pre_rem": 73, "nr_anns_post_rem": 34, "removal_time": "1:38.73"},
    10: {"image_index_range": (90, 99), "nr_anns_pre_rem": 66, "nr_anns_post_rem": 45, "removal_time": "1:06.29"}
}
# Example:
df = avg_removal_time_per_bbox_batches(Path("Timing Experiments Data/tomatoes/annotations/instances_timing.json"), CORRECTION_BATCHES)
df


,batch_key,image_index_range,total_objects,avg_objects_per_image,nr_anns_pre_rem,nr_anns_post_rem,review_time_overhead,avg_removal_time,running_rse_pct
0,5,"(40, 49)",76,7.6,82,57,27.33,2.37,0.00
1,6,"(50, 59)",64,6.4,86,51,28.67,2.29,1.75
2,7,"(60, 69)",67,6.7,101,44,33.67,1.83,7.81
3,8,"(70, 79)",74,7.4,90,50,30.00,1.79,7.34
4,9,"(80, 89)",51,5.1,73,34,24.33,1.91,5.99
5,10,"(90, 99)",72,7.2,66,45,22.00,2.11,4.89


## Evaluate result apples

In [12]:
ANNOTATION_BATCHES = {
    1: {"image_index_range": (0, 9), "annotation_time": "14:03.80"},
    2: {"image_index_range": (10, 19), "annotation_time": "12:20.63"},
    3: {"image_index_range": (20, 29), "annotation_time": "11:32.16"},
    4: {"image_index_range": (30, 39), "annotation_time": "11:51.18"},
}
df = avg_annotation_time_per_object_batches(Path("Timing Experiments Data/apples/annotations/instances_timing.json"), ANNOTATION_BATCHES)
df

,batch_key,image_index_range,total_objects,avg_objects_per_image,min_objects_in_image,max_objects_in_image,total_annotation_time,avg_seconds_per_object,running_rse_pct
0,1,"(0, 9)",156,15.6,12,20,14:03.80,5.41,0.00
1,2,"(10, 19)",129,12.9,12,14,12:20.63,5.74,2.98
2,3,"(20, 29)",127,12.7,12,14,11:32.16,5.45,1.89
3,4,"(30, 39)",136,13.6,12,20,11:51.18,5.23,1.94


In [13]:
CORRECTION_BATCHES = {
    5: {"image_index_range": (40, 49), "nr_anns_pre_rem": 140, "nr_anns_post_rem": 137, "removal_time": "0:53.43"},
    6: {"image_index_range": (50, 59), "nr_anns_pre_rem": 147, "nr_anns_post_rem": 146, "removal_time": "0:50.60"},
    7: {"image_index_range": (60, 69), "nr_anns_pre_rem": 139, "nr_anns_post_rem": 137, "removal_time": "0:59.25"},
    8: {"image_index_range": (70, 79), "nr_anns_pre_rem": 122, "nr_anns_post_rem": 122, "removal_time": "0:35.10"},
    9: {"image_index_range": (80, 89), "nr_anns_pre_rem": 127, "nr_anns_post_rem": 127, "removal_time": "0:38.92"},
    10: {"image_index_range": (90, 99), "nr_anns_pre_rem": 142, "nr_anns_post_rem": 141, "removal_time": "0:51.16"}
}
# Example:
df = avg_removal_time_per_bbox_batches(Path("Timing Experiments Data/apples/annotations/instances_timing.json"), CORRECTION_BATCHES)
df

,batch_key,image_index_range,total_objects,avg_objects_per_image,nr_anns_pre_rem,nr_anns_post_rem,review_time_overhead,avg_removal_time,running_rse_pct
0,5,"(40, 49)",137,13.7,140,137,46.67,2.25,0.00
1,6,"(50, 59)",145,14.5,147,146,49.00,1.60,16.98
2,7,"(60, 69)",138,13.8,139,137,46.33,6.46,44.28
3,8,"(70, 79)",122,12.2,122,122,40.67,0.00,44.28
4,9,"(80, 89)",128,12.8,127,127,42.33,0.00,44.28
5,10,"(90, 99)",140,14.0,142,141,47.33,3.83,30.57
